# Big Data Analysis of Google Play Store Applications / Group 22
###### Team Members: Lotta Kauppinen, Jenna Kiviaho, Marjaana Koski, Jani Laakso, Aleksi Savukoski

#### The chosen dataset contains information of more than 2.3 million applications from Google Play Store. The main goal of this analysis is to find different patterns between the variables. By doing that, we are able to analyze how the different characteristics affect the popularity and ratings.

#### The analysis also aims to showcase if the free apps outperform the paid apps.

#### The other goal for this analysis is to showcase how Spark and MongoDB work together in a Big Data Analysis.


In [265]:
#  Spark Session:

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("GooglePlayStoreAnalysis").getOrCreate()

In [266]:
# Downloading the dataset:
df = spark.read.csv("data/Google-Playstore.csv",header=True,inferSchema=True)
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Rating Count: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Minimum Installs: string (nullable = true)
 |-- Maximum Installs: string (nullable = true)
 |-- Free: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Developer Website: string (nullable = true)
 |-- Developer Email: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Privacy Policy: string (nullable = true)
 |-- Ad Supported: string (nullable = true)
 |-- In App Purchases: string (nullable = true)
 |-- Editors Choice: string (nullable = true)
 |-- Scra

In [267]:
# The amount of rows and columns we are working with:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 2312944
Columns: 24


### Cleaning the data

In [268]:
# Checking the amount of null values by column:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|App Name|App Id|Category|Rating|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price|Currency|Size|Minimum Android|Developer Id|Developer Website|Developer Email|Released|Last Updated|Content Rating|Privacy Policy|Ad Supported|In App Purchases|Editors Choice|Scraped Time|
+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|       0|     0|       0| 22883|       22883|     107|             107|               0|   0|    0|     135| 196|           6530|      

#### Dropping columns that are not needed

In [269]:
# Due to the purpose of this analysis, we are not going to need the information of the columns: 
# Currency (USD), Developer Website, Developer Email, Privacy Policy, Scraped Time, so let's drop them.
df = df.drop("Currency","Developer Website","Developer Email","Privacy Policy","Scraped Time")

#### Handling NULL-values

In [283]:
# Let's check the remaining nulls and make a plan to handle them.
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------+--------+------------+--------+----------------+----------------+----+-----+-----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+----------+
|App Name|App Id|Category|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price| Size|Minimum Android|Developer Id|Released|Last Updated|Content Rating|Ad Supported|In App Purchases|Editors Choice|Rating_num|
+--------+------+--------+------------+--------+----------------+----------------+----+-----+-----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+----------+
|       0|     0|       0|           0|     133|             107|               0|  30|   19|75005|           6530|          33|   71055|           0|             0|           4|               2|             0|         0|
+--------+------+--------+------------+--------+----------------+----------------+----+-----+-----+-------------

#### Rating and Rating Count -columns: NULL-values
##### These columns includes the average rating and number of rating of each app. All of the NULL-values and values that are not numeric will be changed to 0.0 (Rating) and 0 (Rating Count).

In [271]:
from pyspark.sql.functions import col

# Let's find the rows that include values that ar NOT int or float.
non_numeric = df.filter(~col("Rating").rlike(r'^-?\d+(\.\d+)?$'))

# Examples of the values that are NOT int or float.
non_numeric.select("Rating").show(100, truncate=False)


+--------------------------------+
|Rating                          |
+--------------------------------+
|Books & Reference               |
|com.andromo.dev807396.app1066982|
|net.jp.apps.hirotsuchiya.kanji  |
|Education                       |
|com.storyshare.wisdom           |
|Shopping                        |
|Productivity                    |
|com.app.aziuyr                  |
|Educational                     |
|Social                          |
|com.menshchyna.android          |
|kr.manse.catdog                 |
|com.heliconia.estudio           |
|Education                       |
|Productivity                    |
|Shopping                        |
|Productivity                    |
|Food & Drink                    |
|Racing                          |
|com.ksmps.cricketalpha          |
|Business                        |
|Entertainment                   |
|News & Magazines                |
|Board                           |
|Finance                         |
|Entertainment      

In [272]:
from pyspark.sql.functions import col, when, replace

# It seems that some of the App-values have been written in the wrong columns. Let's fill all the non-numeric values with 0.0.
df = df.withColumn(
    "Rating_num",
    when(col("Rating").rlike(r'^-?\d+(\.\d+)?$'), col("Rating").cast("float"))
    .otherwise(0.0)
)

# To check that the code worked:
df.select("Rating", "Rating_num").filter(col("Rating_num") == 0.0).show(100, truncate=False)

+------+----------+
|Rating|Rating_num|
+------+----------+
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |


In [273]:
df = df.withColumn(
    "Rating Count", 
    when(col("Rating_num") == 0.0, "0")  # Rating_num == 0.0 > Rating Count = 0
    .otherwise(col("Rating Count"))      
)

# To check that the code worked:
df.select("Rating", "Rating_num", "Rating Count").filter(col("Rating_num") == 0.0).show(100, truncate=False)
df = df.drop("Rating")

+------+----------+------------+
|Rating|Rating_num|Rating Count|
+------+----------+------------+
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.

### Changing the data types

In [275]:
# Checking the data types
print(df.schema["Installs"].dataType)
print(df.schema["Rating_num"].dataType)
print(df.schema["Price"].dataType)
print(df.schema["Rating Count"].dataType)
print(df.schema["Size"].dataType)
print(df.schema["Free"].dataType)

StringType()
DoubleType()
StringType()
StringType()
StringType()
StringType()


In [276]:
from pyspark.sql.types import IntegerType, FloatType, BooleanType
from pyspark.sql.functions import translate, when, round, expr, lower

# Changing data types
# Using try_cast instead of cast for unexpected values
df = df.withColumn("Installs", expr("try_cast(translate(Installs, '+,', '') as int)")) # Removing characters '+' and ','
df = df.withColumn("Price", expr("try_cast(Price as float)"))
df = df.withColumn("Rating Count", expr("try_cast(`Rating Count` as int)"))
df = df.withColumn("Free", 
    when(lower(col("Free").cast("string")) == "true", True)
    .when(lower(col("Free").cast("string")) == "false", False)
    .otherwise(None)
)

# Handling column "Size"
df = df.withColumn("Size", 
    when(col("Size").contains("k"),
         expr("try_cast(translate(Size, 'k,', ' .') as float)") / 1024) # Removing 'k' and ',', then divided by 1024 so every size is in MB
    .when(col("Size").contains("M"), 
         expr("try_cast(translate(Size, 'M,', ' .') as float)")) # Removing characters 'M' and ','
    .otherwise(expr("try_cast(Size as float)")) # If it's only a number and no characters
)

# Rounding column "Size"
df = df.withColumn("Size", round(col("Size"), 2))

# Checking data types again to make sure they are correct
print(df.schema["Installs"].dataType)
print(df.schema["Price"].dataType)
print(df.schema["Rating Count"].dataType)
print(df.schema["Free"].dataType)
print(df.schema["Size"].dataType)


IntegerType()
FloatType()
IntegerType()
BooleanType()
DoubleType()
